In [ ]:
#pip install sentence-transformers chromadb requests

import os
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from chromadb.config import Settings
import requests
import chromadb
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1. Load embedding model
# -----------------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# 2. Initialize Chroma DB
# -----------------------------
chroma_client = chromadb.PersistentClient(path="C:\\rag-demo\\rag_chroma_db")


collection = chroma_client.get_or_create_collection(
    name="rag_docs",
    metadata={"hnsw:space": "cosine"}
)

# -----------------------------
# 3. Add documents
# -----------------------------
docs = [
    "Azure Blob Storage is a massively scalable object store for unstructured data.",
    "Azure Data Factory is a cloud-based ETL and data orchestration service.",
    "RAG systems combine retrieval and generation to ground LLM responses."
]

embeddings = embedder.encode(docs).tolist()

collection.add(
    documents=docs,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(docs))]
)

# -----------------------------
# 4. Retrieval function
# -----------------------------
def retrieve(query, k=3):
    q_emb = embedder.encode([query]).tolist()[0]
    results = collection.query(query_embeddings=[q_emb], n_results=k)
    return results["documents"][0]

# -----------------------------
# 5. Call Ollama for generation
# -----------------------------
def call_ollama(prompt, model="llama3"):
   # print("Prompt sent to Ollama:", prompt)

    response = requests.post("http://localhost:11434/api/generate",json={"model": "llama3", "prompt": prompt, "stream": False})
    return response.json()

# -----------------------------
# 6. Full RAG pipeline
# -----------------------------
def rag(query):
    retrieved_docs = retrieve(query)
    context = "\n".join(retrieved_docs)

    prompt = f"""
You are a helpful assistant. Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

Answer:
"""

    return call_ollama(prompt)

# -----------------------------
# 7. Test it
# -----------------------------
print(rag("What is Azure Blob Storage used for?"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11793.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'model': 'llama3', 'created_at': '2026-04-03T03:39:40.9486126Z', 'response': 'According to the context, Azure Blob Storage is a massively scalable object store for unstructured data.', 'done': True, 'done_reason': 'stop', 'context': [128006, 882, 128007, 1432, 2675, 527, 264, 11190, 18328, 13, 5560, 27785, 279, 2317, 3770, 311, 4320, 382, 2014, 512, 79207, 50539, 15035, 374, 264, 64807, 69311, 1665, 3637, 369, 653, 52243, 828, 627, 79207, 2956, 17367, 374, 264, 9624, 6108, 469, 13778, 323, 828, 70984, 2214, 2532, 627, 49, 1929, 6067, 16343, 57470, 323, 9659, 311, 5015, 445, 11237, 14847, 382, 14924, 512, 3923, 374, 35219, 50539, 15035, 1511, 369, 1980, 16533, 512, 128009, 128006, 78191, 128007, 271, 11439, 311, 279, 2317, 11, 35219, 50539, 15035, 374, 264, 64807, 69311, 1665, 3637, 369, 653, 52243, 828, 13], 'total_duration': 5986561200, 'load_duration': 291421500, 'prompt_eval_count': 80, 'prompt_eval_duration': 3498933200, 'eval_count': 20, 'eval_duration': 2129517500}


In [4]:
import requests

# url = "http://localhost:11434/api/generate"
# payload = {"model": "llama3", "prompt": "Hello!"}

# with requests.post(url, json=payload, stream=True) as r:
#     for line in r.iter_lines():
#         if line:
#             data = line.decode("utf-8")
#             print(data)




resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "llama3", "prompt": "Do you know about Cat and Dogs ?", "stream": False}
)

print(resp.json())




{'model': 'llama3', 'created_at': '2026-04-03T03:40:11.5203541Z', 'response': 'A popular topic!\n\nYes, I\'m familiar with cats and dogs. Both are popular household pets that have been domesticated for thousands of years.\n\n**Cats:**\n\n* Also known as Felis catus\n* Known for their agility, playful personalities, and independence\n* Typically solitary animals, but can form strong bonds with their human caregivers\n* Come in a variety of breeds, sizes, and coat lengths\n* Known for their grooming habits, sharp claws, and distinctive vocalizations (meowing)\n\n**Dogs:**\n\n* Also known as Canis lupus familiaris\n* Known for their loyalty, playful personalities, and ability to form strong bonds with humans\n* Often referred to as "man\'s best friend" due to their long history of domestication and companionship\n* Come in a wide range of breeds, sizes, and coat types\n* Known for their ability to assist with tasks such as hunting, herding, and service work\n\nBoth cats and dogs can make 